# Data Assimilaiton on a reduced order model

In [ ]:
import numpy as np
from utils import  load_cylinder_dataset

X_true, X_noisy, simulation_dir = load_cylinder_dataset(noise_type='gauss', 
                                                        noise_level=0.02, 
                                                        smoothing=0.01,
                                                        visualize=True,
                                                        )

In [ ]:

# # ## 5. Create Observations from test flowfields
# # Project the true and noisy test flow fields onto the trained POD modes,
# # then wrap them in an `Observations` object for downstream data assimilation.

# # %%
# from observations import Observations

# Nt_test = X_test.shape[1]
# t_test  = np.arange(Nt_test) * dt

# # Flatten test flow fields onto fluid points using the same NaN mask as training
# # (subtract_mean=False because project_data_onto_Psi subtracts the training mean)
# Q_test_true,  _, _ = prepare_data([*X_test_true], subtract_mean=False)
# Q_test_noisy, _, _ = prepare_data([*X_test],      subtract_mean=False)

# # Project onto POD modes → temporal coefficients (Nt_test, N_modes)
# Phi_test_true  = case_ESN.project_data_onto_Psi(Q_test_true,  remove_mean=True)
# Phi_test_noisy = case_ESN.project_data_onto_Psi(Q_test_noisy, remove_mean=True)

# obs = Observations(
#     model=None,
#     y_true=Phi_test_true,
#     y_raw=Phi_test_noisy,
#     t_true=t_test,
#     t_start=t_test[0],
#     t_stop=t_test[-1],
#     Nt_obs=20,
# )

# print(f'y_obs shape : {obs.y_obs.shape}  ({len(obs.t_obs)} observation times)')


In [ ]:
print(f'Dimension of the data:  {X_true.shape}= (N_samples, Nx, Ny, N_vars). \nTotal d.o.f. = {np.prod(X_true.shape[1:])}.')

## The POD-ESN Model

The POD-ESN combines **Proper Orthogonal Decomposition (POD)** for spatial compression with an **Echo State Network (ESN)** for temporal forecasting, enabling efficient real-time modelling of high-dimensional flow fields.

---

### Step 1: Dimensionality Reduction via POD

The flow field $\mathbf{u}(\mathbf{x}, t) \in \mathbb{R}^{N_u}$ is projected onto the $N_m \ll N_u$ most energetic spatial modes:

$$\mathbf{u}(\mathbf{x}, t) \approx \boldsymbol{\Psi}(\mathbf{x})\,\boldsymbol{\phi}(t) = \sum_{i=1}^{N_m} \phi_i(t)\, \boldsymbol{\psi}_i(\mathbf{x})$$

where $\boldsymbol{\Psi} = [\boldsymbol{\psi}_1 \mid \cdots \mid \boldsymbol{\psi}_{N_m}] \in \mathbb{R}^{N_u \times N_m}$ are the POD modes and $\boldsymbol{\phi}(t) \in \mathbb{R}^{N_m}$ are the time-varying POD coefficients. The ESN only needs to model the low-dimensional $\boldsymbol{\phi}(t)$, not the full field.

---

### Step 2: Temporal Modelling via ESN

The ESN lifts $\boldsymbol{\phi}_k$ into a high-dimensional **reservoir state** $\mathbf{r}_k \in \mathbb{R}^{N_r}$, with $N_r \gg N_m$, whose dynamics are governed by the **forced dynamical system**

$$\mathbf{r}_{k+1} = \tanh\!\Big(\underbrace{\sigma_\mathrm{in}\,\mathbf{W}_\mathrm{in}\,\boldsymbol{\phi}_k}_{\text{forcing}} + \underbrace{\rho\,\mathbf{W}\,\mathbf{r}_k}_{\text{memory}}\Big)$$
where the POD coefficients act as an instantaneous forcing term and the reservoir state carries memory of past dynamics. The forecast POD coefficients are recovered via a **linear readout**:

$$\hat{\boldsymbol{\phi}}_{k+1} = \mathbf{W}_\mathrm{out}\,\mathbf{r}_{k+1}$$

The three components of the ESN are:

- **Input layer** $\mathbf{W}_\mathrm{in} \in \mathbb{R}^{N_r \times N_m}$ — randomly initialised and **fixed**; controlled by the input scaling $\sigma_\mathrm{in}$, which tunes $\tanh$ saturation and nonlinearity
- **Reservoir** $\mathbf{W} \in \mathbb{R}^{N_r \times N_r}$ (sparse) — randomly initialised and **fixed**; rescaled so its spectral radius equals $\rho < 1$, which controls memory length and ensures the *echo state property* (the influence of past states vanishes gradually)
- **Readout** $\mathbf{W}_\mathrm{out} \in \mathbb{R}^{N_m \times N_r}$ — the **only trained matrix**, obtained in one shot via ridge regression:

$$\left(\mathbf{R}\mathbf{R}^\mathrm{T} + \gamma\,\mathbf{I}\right)\mathbf{W}_\mathrm{out}^\mathrm{T} = \mathbf{R}\,\boldsymbol{\Phi}^\mathrm{T}$$

where  $\boldsymbol{\Phi} \in \mathbb{R}^{N_m \times N}$ contains the training POD coefficients, which projection onto the reservoir is collected in $\mathbf{R} \in \mathbb{R}^{N_r \times N}$, and $\gamma$ is the Tikhonov regularisation parameter. This is a **convex problem with a unique global minimum**.
<br>
<br>
<br>

<a><img src="../../../docs/figs/DA/ESN-open-close-schematics.png" width=1000px></a>



# Part I. Building the POD-ESN model


In [ ]:
### Separate the dataset into data to train the POD-ESN and data to test the POD-ESN-EnKF

N_train = 40
N_val = 20

X_train, X_train_true = [yy[:N_train+N_val].T for yy in [X_noisy, X_true]]
X_filter, X_filter_true = [yy[N_train+N_val:].T for yy in [X_noisy, X_true]]

dt = 0.01
t_val = N_val * dt 
t_train = N_train * dt 



In [ ]:
from models.data_driven import POD_ESN


case_POD_ESN = POD_ESN(data=X_train,
                       
                    # ====== POD arguments ======== # 
                    domain=[-2.5, 2.5, 0, 12],  
                    N_modes=4,  
                    skip_sensor_placement=True,
                    
                    # ====== ESN arguments ======== # 
                    t_val=t_val,
                    t_train=t_train,
                    t_test=0.,
                    N_units=40, 
                    train_ESN=True,
                    N_wash=5,
                    noise=1E-4,
                    N_func_evals=26,
                    rho_range=[0.2, 0.9],
                    upsample=2,            
                    Nq=2, 
                  
                    # ====== Model arguments ======== # 
                    dt = dt,
                    figs_folder=simulation_dir, 
                    plot_case=True
                    )


In [ ]:
# Check the forecast of the POD-ESN

psi, t = case_POD_ESN.time_integrate(Nt=1000)
case_POD_ESN.update_history(psi, t)

case_POD_ESN.visualize_state_hist(max_modes=case_POD_ESN.N_modes, 
                                  t_zoom=N_train)

 
# Part II. State estimation via data assimilation

<a><img src="../../../docs/figs/DA/POD-ESN_DA.png" width=800px></a>


How many datapoints do we want to assimilate?

In [ ]:
case_POD_ESN.select_sensors(domain_of_measurement=[-1.5, 1.5, 4, 8], 
                            N_sensors=2, 
                            qr_selection=True);

case_POD_ESN.plot_flows_rms(case=case_POD_ESN, datasets=[X_filter], names=['Test data'], 
                            display_sensors=True);

In [ ]:
from observations import Observations


N_test = X_filter.shape[-1]

data_obs = X_filter.copy().reshape(-1, N_test)[case_POD_ESN.sensor_locations].T
data_obs_true = X_filter_true.copy().reshape(-1, N_test)[case_POD_ESN.sensor_locations].T

dt = case_POD_ESN.dt
t_true = np.arange(0, N_test) * dt

t_start = 0.5
t_stop = min(5., t_true[-10])

truth = Observations(
    y_raw=data_obs,          # shape (N_test, Nq)
    y_true=data_obs_true,    # shape (N_test, Nq)
    t_true=t_true,
    t_start=t_start,
    t_stop=t_stop,
    Nt_obs=0.25 // dt, #time steps between two observations
    )


Observations.plot_truth(truth, window=5, fig_width=10)

#### Initialize the ensemble 



In [ ]:
from ensemble import Ensemble
from data_assimilation import EnSRKF

ensemble = Ensemble(parent_model=case_POD_ESN,
                    da_method=EnSRKF,
                    m=50,
                    std_phi=1.1)

In [ ]:
import numpy as np

filter_ens = ensemble.copy()

std_obs = .01 * np.max(truth.t_obs) # observation error standard deviation
Cdd = np.eye(filter_ens.model.Nq) * std_obs**2

for d, t_d in zip(truth.y_obs, truth.t_obs):
    filter_ens.forecast_step(t_end=t_d)
    filter_ens.analysis_step(d=d, Cdd=Cdd)
    filter_ens.assimilated_data = (d, t_d)


In [ ]:

filter_ens.visualize_history(plot_members=True, 
                             truth=truth)


# Part II. State & Parameter estimation: online learning of Wout
Why us the SVD interesting in this scenario? Because we can use data assimialtion to modify on the fly the value of the singluar values. This is, we can perform state and parameter inference with the ESN on the fly. In machine learning jargon, this is known as *online learning*.

With this objective, we first need to create an ensemble of POD-ESN with different SVDs for each ensemble member. This is achieved by providing `Wout` as the parameter to estimate, i.e., as the `est_a` argument. The `sta_a` parameter indicates the uncewrtainty around the Wout SVDs.


## Singular value decomposition of Wout
We can decompose the trained Wout into its singular value components using the library ```scipy.linalg```. The multiplication of the three matrices are equivalent to the original Wout. 

In [ ]:
import scipy.linalg as sla
import matplotlib.pyplot as plt


# Compute the SVD of the output matrix
[Wout_U, Wout_eigs, Wout_Vh] = sla.svd(ensemble.model.Wout, full_matrices=False)


# #display the three matrices
fig, axs = plt.subplots(1, 4, figsize=(15, 15), width_ratios=[1, 1, 1, 1])
for W, ax, title in zip([ensemble.model.Wout, Wout_U, np.diag(Wout_eigs), Wout_Vh], axs, 
                        ['$\\mathbf{W_{out}} = $', '$\\mathbf{U}$', '$\\Sigma$', '$\\mathbf{V}^\\mathrm{T}$']):
    im = ax.imshow(W, cmap='PuOr', vmin=-np.max(W), vmax=np.max(W))
    ax.set(title=title)
    # set the same colorbar for all the matrices
    fig.colorbar(im, ax=ax, shrink=.9, orientation='horizontal')

print('Dimensions of matrices:', Wout_U.shape, Wout_eigs.shape, Wout_Vh.shape)

assert np.allclose(ensemble.model.Wout, np.dot(Wout_U, np.dot(np.diag(Wout_eigs), Wout_Vh)))


## State and parameter estimation on the POD-ESN

<a><img src="../../../docs/figs/DA/POD-ESN-SPE.png" width=800px></a>



In [ ]:
# Define ensemble


ensemble_2 = Ensemble(parent_model=case_POD_ESN,
                    da_method=EnSRKF,
                    m=50,
                    std_phi=.1,
                    est_alpha=['Wout'], # Estimate the Wout singular values
                    std_alpha=0.02       # Standard deviation of the noise added to the Wout singular values
                    )

### Wout as SVD
Now the output matrix is decomposed in 3 matrices. the diagonal of the central one is the matrix of singular valuess. these we can modify in time via real-time DA.
Uncertainty in the parameters is also shown as increased uncertainty in the states

In [ ]:
case = ensemble_2.model.copy()
psi, t = case.time_integrate(Nt=500)
case.update_history(psi, t)
case.visualize_observable_hist(t_zoom=N_train)

In [ ]:

truth = Observations(
    y_raw=data_obs,          
    y_true=data_obs_true,    
    t_true=np.arange(0, N_test) * case_POD_ESN.dt,
    t_start=0.5,
    t_stop=10.,
    Nt_obs=0.15 // case_POD_ESN.dt, #time steps between two observations
    )


In [ ]:
import numpy as np

filter_ens = ensemble_2.copy()
filter_ens.update_history(psi[-1], 0.0, reset=True)

std_obs = .01 * np.max(truth.t_obs) # observation error standard deviation
Cdd = np.eye(filter_ens.model.Nq) * std_obs**2

for d, t_d in zip(truth.y_obs, truth.t_obs):
    filter_ens.forecast_step(t_end=t_d)
    filter_ens.analysis_step(d=d, Cdd=Cdd)
    filter_ens.assimilated_data = (d, t_d)

filter_ens.visualize_history(plot_members=True, 
                             truth=truth)


In [ ]:
# Visualize the flow field reconstruction
from utils import animate_flowfields

case = filter_ens.model.copy() #type: POD_ESN

time = case.hist_t
hist = case.hist[:, :case.N_modes, :]

prediction_ensemble =[]

for mi in range(filter_ens.m):
    if mi > 10:
        break
    prediction_mi = case.reconstruct(Phi=hist[..., mi])
    prediction_ensemble.append(prediction_mi)

prediction_ensemble = np.array(prediction_ensemble)
prediction_mean = np.mean(prediction_ensemble, axis=0)
prediction_std = np.std(prediction_ensemble, axis=0) 


test_data = X_filter.copy()[..., :len(time)]
test_data_true = X_filter_true.copy()[..., :len(time)]

# Compute Error metrics
RMS_noisy = np.array([POD_ESN.compute_RMS(test_data[...,ti], prediction_mean[...,ti]) 
                      for ti in range(prediction_mean.shape[-1])]).transpose(1, 2,3, 0) # reshape to (Nu, Nx, Ny, Nt)


In [ ]:
sensors = case.sensor_locations[case.sensor_locations < test_data.shape[1] * test_data.shape[2]]
x_col = sensors // test_data.shape[2]
y_row = sensors % test_data.shape[2]



U_datasets = {'Truth': test_data[0], 
              'POD_ESN mean': prediction_mean[0], 
              'POD_ESN std': prediction_std[0], 
              'RMS Error': RMS_noisy[0]}

# compute the observation indices in the downsampled time history
anim = animate_flowfields(U_datasets, 
                          time=time, 
                          n_frames=50, figsize=(8, 6), step=5,
                          assimilated_data={'t_obs': truth.t_obs,   # observation times
                                            'xy': np.column_stack((x_col, y_row))  # sensor locations
                                            }
                          )

In [ ]:
from IPython.display import HTML
HTML(anim.to_jshtml())